In [ ]:
# ============================================================
# FINAL FIXED ShuffleNet V2 — FloodNet (NO DATA BUGS)
# ============================================================

import os, glob, time, torch, numpy as np
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using:", DEVICE)

# ── DATA PATH ──
BASE = '/kaggle/input/datasets/aletbm/aerial-imagery-dataset-floodnet-challenge/FloodNet Challenge - Track 1'

def get_images(folder):
    return glob.glob(os.path.join(folder, "*.jpg"))

# ── LOAD TRAIN DATA ──
flooded = [(p,1) for p in get_images(os.path.join(BASE,'Train/Labeled/Flooded/image'))]
nonflooded = [(p,0) for p in get_images(os.path.join(BASE,'Train/Labeled/Non-Flooded/image'))]

train_labeled = flooded + nonflooded
unlabeled = get_images(os.path.join(BASE,'Train/Unlabeled/image'))

print("Total labeled:", len(train_labeled), "Unlabeled:", len(unlabeled))

# ── FIX 1: CREATE VALIDATION SPLIT (NO EMPTY VAL) ──
train_labeled, val_data = train_test_split(
    train_labeled,
    test_size=0.2,
    stratify=[l for _,l in train_labeled],
    random_state=42
)

print("Train:", len(train_labeled), "Val:", len(val_data))

# ── DATASET ──
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class LabeledDataset(Dataset):
    def __init__(self, samples): self.samples=samples
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p,l = self.samples[i]
        img = transform(Image.open(p).convert('RGB'))
        return img, torch.tensor(l,dtype=torch.float32)

class UnlabeledDataset(Dataset):
    def __init__(self, paths): self.paths=paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = transform(Image.open(self.paths[i]).convert('RGB'))
        return img

# ── SAMPLER ──
def make_sampler(samples):
    labels=[l for _,l in samples]
    counts=np.bincount(labels)
    weights=1./counts
    sample_weights=[weights[l] for l in labels]
    return WeightedRandomSampler(sample_weights,len(sample_weights))

# ── LOADERS ──
train_loader = DataLoader(LabeledDataset(train_labeled), batch_size=16, sampler=make_sampler(train_labeled))
val_loader   = DataLoader(LabeledDataset(val_data), batch_size=16)
unlabeled_loader = DataLoader(UnlabeledDataset(unlabeled), batch_size=16)

# ── MODEL (FIXED) ──
model = models.shufflenet_v2_x1_0(weights=models.ShuffleNet_V2_X1_0_Weights.IMAGENET1K_V1)

# ❌ REMOVE Sigmoid → FIXED
model.fc = nn.Linear(model.fc.in_features,1)

model = model.to(DEVICE)

# ── FIX 2: CORRECT LOSS ──
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# ── FIX 3: SAFE PSEUDO LABELING ──
LAMBDA = 0.2

def get_pseudo(model):
    model.eval()
    pseudo=[]
    with torch.no_grad():
        for imgs in unlabeled_loader:
            imgs=imgs.to(DEVICE)
            probs=torch.sigmoid(model(imgs)).squeeze(1)
            for i,p in enumerate(probs):
                if p < 0.5 - LAMBDA:
                    pseudo.append((imgs[i].cpu(),0))
                elif p > 0.5 + LAMBDA:
                    pseudo.append((imgs[i].cpu(),1))
    return pseudo

# ── EVALUATION ──
def evaluate(loader):
    model.eval()
    preds,labels,probs_all=[],[],[]
    with torch.no_grad():
        for imgs,l in loader:
            imgs=imgs.to(DEVICE)
            p=torch.sigmoid(model(imgs)).squeeze(1)
            preds+=(p>0.5).cpu().numpy().tolist()
            probs_all+=p.cpu().numpy().tolist()
            labels+=l.numpy().tolist()

    if len(set(labels)) < 2:
        return {"acc":0,"f1":0,"roc":0}

    return {
        "acc": accuracy_score(labels,preds)*100,
        "f1": f1_score(labels,preds)*100,
        "roc": roc_auc_score(labels,probs_all)*100
    }

# ── TRAIN ──
EPOCHS=30
best_f1=0
history=[]

for epoch in range(EPOCHS):
    model.train()
    loss_total=0

    # supervised
    for imgs,labels in train_loader:
        imgs,labels=imgs.to(DEVICE),labels.to(DEVICE)
        optimizer.zero_grad()
        loss=criterion(model(imgs).squeeze(1),labels)
        loss.backward()
        optimizer.step()
        loss_total+=loss.item()

    # pseudo
    if epoch>10:
        pseudo=get_pseudo(model)
        if len(pseudo)>0:
            imgs=torch.stack([p[0] for p in pseudo]).to(DEVICE)
            labels=torch.tensor([p[1] for p in pseudo],dtype=torch.float32).to(DEVICE)

            for i in range(0,len(imgs),16):
                optimizer.zero_grad()
                loss=criterion(model(imgs[i:i+16]).squeeze(1),labels[i:i+16])
                loss.backward()
                optimizer.step()

    val=evaluate(val_loader)
    history.append(val["acc"])

    if val["f1"]>best_f1:
        best_f1=val["f1"]
        torch.save(model.state_dict(),"/kaggle/working/best_shufflenet.pth")

    print(f"Epoch {epoch+1} | Loss {loss_total:.3f} | Acc {val['acc']:.2f} | F1 {val['f1']:.2f}")

# ── PLOT ──
plt.plot(history)
plt.title("Validation Accuracy")
plt.show()

print("✅ DONE")

Using: cuda
Total labeled: 398 Unlabeled: 1047
Train: 318 Val: 80
Downloading: "https://download.pytorch.org/models/shufflenetv2_x1-5666bf0f80.pth" to /root/.cache/torch/hub/checkpoints/shufflenetv2_x1-5666bf0f80.pth


100%|██████████| 8.79M/8.79M [00:00<00:00, 100MB/s]


Epoch 1 | Loss 13.702 | Acc 95.00 | F1 75.00
Epoch 2 | Loss 13.090 | Acc 97.50 | F1 88.89
Epoch 3 | Loss 12.122 | Acc 97.50 | F1 88.89
Epoch 4 | Loss 11.004 | Acc 97.50 | F1 88.89
Epoch 5 | Loss 9.944 | Acc 97.50 | F1 88.89
Epoch 6 | Loss 8.698 | Acc 97.50 | F1 88.89
Epoch 7 | Loss 7.669 | Acc 96.25 | F1 82.35
Epoch 8 | Loss 6.919 | Acc 96.25 | F1 82.35
Epoch 9 | Loss 6.103 | Acc 96.25 | F1 82.35
Epoch 10 | Loss 5.501 | Acc 96.25 | F1 82.35
